In [ ]:
import json
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup


In [ ]:
options = Options()
# options.add_argument("--headless=new")
driver = webdriver.Chrome(options=options)

url = "https://aiche.confex.com/aiche/2023/meetingapp.cgi/ModuleProgramBook/0?clearcache=1"
driver.get(url)

# Wait specifically for the nested target to exist in the DOM
WebDriverWait(driver, 60).until(
    EC.presence_of_element_located((
        By.CSS_SELECTOR,
        "main#main section#details section.pageContent section.field_ChildList_Program"
    ))
)

soup = BeautifulSoup(driver.page_source, "html.parser")

# Find: main#main > section#details > section.pageContent > section.field_ChildList_Program
target = soup.select_one(
    "main#main section#details section.pageContent section.field_ChildList_Program"
)

if not target:
    print("❌ Target section not found.")
else:
    # Print the visible text
    print(target.get_text("\n", strip=True))

    # (Optional) also save the raw HTML to a file for debugging
    with open("field_ChildList_Program.html", "w", encoding="utf-8") as f:
        f.write(target.prettify())

driver.quit()

TimeoutException: Message: 
Stacktrace:
	GetHandleVerifier [0x0x7ff748efe415+77285]
	GetHandleVerifier [0x0x7ff748efe470+77376]
	(No symbol) [0x0x7ff748cc9a6a]
	(No symbol) [0x0x7ff748d20406]
	(No symbol) [0x0x7ff748d206bc]
	(No symbol) [0x0x7ff748d73ac7]
	(No symbol) [0x0x7ff748d4864f]
	(No symbol) [0x0x7ff748d7087f]
	(No symbol) [0x0x7ff748d483e3]
	(No symbol) [0x0x7ff748d11521]
	(No symbol) [0x0x7ff748d122b3]
	GetHandleVerifier [0x0x7ff7491e1efd+3107021]
	GetHandleVerifier [0x0x7ff7491dc29d+3083373]
	GetHandleVerifier [0x0x7ff7491fbedd+3213485]
	GetHandleVerifier [0x0x7ff748f1884e+184862]
	GetHandleVerifier [0x0x7ff748f2055f+216879]
	GetHandleVerifier [0x0x7ff748f07084+113236]
	GetHandleVerifier [0x0x7ff748f07239+113673]
	GetHandleVerifier [0x0x7ff748eee298+11368]
	BaseThreadInitThunk [0x0x7ff8f895e8d7+23]
	RtlUserThreadStart [0x0x7ff8f9cfc34c+44]


In [ ]:

# Headless Chrome setup
options = Options()
# options.add_argument("--headless")
driver = webdriver.Chrome(options=options)

url = "https://aiche.confex.com/aiche/2023/meetingapp.cgi/Program/3315"
driver.get(url)

# Wait for the calendar content to load (wait for any CalendarList)
WebDriverWait(driver, 60).until(
    EC.presence_of_element_located((By.CLASS_NAME, "CalendarList"))
)

soup = BeautifulSoup(driver.page_source, "html.parser")

ul_calendar = soup.find("ul", class_="Calendar")

data = []

if ul_calendar:
    calendar_lists = ul_calendar.find_all("ul", class_="CalendarList")

    for cal_list in calendar_lists:
        # Extract date (first li span with class defaultTZ inside .date span)
        date_li = cal_list.find("li")
        date = None
        if date_li:
            date_span = date_li.find("span", class_="defaultTZ")
            if date_span:
                date = date_span.get_text(strip=True)

        # Extract time from <time class="first">
        time_tag = cal_list.find("time", class_="first")
        time_range = time_tag.get_text(strip=True) if time_tag else None

        # Extract sessions inside <section class="itemCalendar ...">
        sessions = []
        session_sections = cal_list.find_all("section", class_="itemCalendar")
        for session in session_sections:
            # Session title and link
            a_tag = session.find("a")
            session_title = a_tag.get_text(strip=True) if a_tag else None
            session_link = a_tag['href'] if a_tag and 'href' in a_tag.attrs else None

            # Speakers - bold tags inside span with class topDisplay
            speakers = []
            top_display = session.find("span", class_="topDisplay")
            if top_display:
                bolds = top_display.find_all("b")
                for b in bolds:
                    speakers.append(b.get_text(strip=True))

            # Location info inside ul.propertyInfo > li.propertyName
            location = None
            prop_info = session.find("ul", class_="propertyInfo")
            if prop_info:
                prop_name = prop_info.find("li", class_="propertyName")
                if prop_name:
                    location = prop_name.get_text(strip=True)

            sessions.append({
                "title": session_title,
                "link": session_link,
                "speakers": speakers,
                "location": location
            })

        data.append({
            "date": date,
            "time": time_range,
            "sessions": sessions
        })

else:
    print("❌ <ul class='Calendar'> not found.")

driver.quit()

# Save JSON data to file
with open("sessions.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print("✅ Data saved to sessions.json")


✅ Data saved to sessions.json


In [ ]:
import json
import os
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time

BASE_URL = "https://aiche.confex.com/aiche/2023/meetingapp.cgi/"
all_links_by_session = {}

def get_all_session_urls(json_file_path):
    with open(json_file_path, "r", encoding="utf-8") as file:
        sessions_data = json.load(file)

    session_urls = []
    for entry in sessions_data:
        for session in entry.get("sessions", []):
            session_link = session.get("link")
            if session_link:
                session_urls.append(BASE_URL + session_link)

    return session_urls


def extract_presentation_links_from_live_page(url):
    options = Options()
    options.add_argument("--headless")
    driver = None

    try:
        driver = webdriver.Chrome(options=options)
        print(f"🔗 Opening URL: {url}")
        driver.get(url)

        # Wait for Presentations section to load

        WebDriverWait(driver, 40).until_not(
                    EC.text_to_be_present_in_element(
                        (By.TAG_NAME, "body"),
                        "please wait while the program loads"
                    )
                )

        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "section.field_ChildList_PaperSlot"))
        )
        time.sleep(5)
        print("✅ Presentations section found.")

        # Additional wait to allow dynamic content to load inside the <ul>
        print("⏳ Waiting 10 seconds for dynamic content inside <ul> to load...")
        time.sleep(10)

        # Wait for at least one <a> inside the <ul.PaperSlot> if possible
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "ul.PaperSlot a[href^='Paper/']"))
        )
        print("✅ At least one paper link found inside <ul.PaperSlot>.")

        soup = BeautifulSoup(driver.page_source, 'html.parser')
        presentation_section = soup.select_one("section.field_ChildList_PaperSlot")
        if not presentation_section:
            print("❌ Presentations section not found after waiting.")
            return []

        presentation_section_ul = presentation_section.find("ul", class_="PaperSlot")
        if not presentation_section_ul:
            print("❌ <ul> with class PaperSlot not found.")
            return []

        links = []
        for a_tag in presentation_section_ul.find_all("a", href=True):
            href = a_tag["href"]
            if href.startswith("Paper/"):
                full_link = BASE_URL + href
                links.append(full_link)

        return links

    except Exception as e:
        print(f"❌ Error: {e}")
        return []

    finally:
        if driver:
            driver.quit()



# --- Run everything ---
def main():
    all_session_urls = get_all_session_urls("sessions.json")
    all_links_by_session = {}

    if not all_session_urls:
        print("❌ No session URLs found.")
        return

    for idx, session_url in enumerate(all_session_urls, start=1):
        print(f"\n🔍 Processing session {idx} / {len(all_session_urls)}: {session_url}")
        links = extract_presentation_links_from_live_page(session_url)
        print(f"✅ Found {len(links)} presentation links.")
        all_links_by_session[session_url] = links

    # Save results to JSON
    output_file = "presentation_links_by_session_3330.json"
    output_dir = os.path.dirname(output_file)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(all_links_by_session, f, indent=2)

    print(f"\n✅ All presentation links saved to {output_file}")

if __name__ == "__main__":
    main()

In [ ]:
import json
import os
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import glob

# --- Setup headless browser ---
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

def get_text(soup, selector):
    elements = soup.select(selector)
    if not elements:
        return "Not found"
    return " | ".join([el.get_text(strip=True) for el in elements])

def load_all_links_from_json(json_file_path):
    if not os.path.exists(json_file_path):
        print(f"❌ File {json_file_path} not found.")
        return []

    with open(json_file_path, "r", encoding="utf-8") as f:
        session_links = json.load(f)

    all_links = []
    for session_url, links in session_links.items():
        all_links.extend(links)
    print(f"✅ Loaded {len(all_links)} total presentation links from {json_file_path}.")
    return all_links

def extract_and_save_presentation_data(links, output_file="aiche_papers.json"):
    if not links:
        print("⚠️ No links to process.")
        return

    driver = webdriver.Chrome(options=options)
    all_data = []

    try:
        for idx, url in enumerate(links, start=1):
            print(f"\n🔍 Processing ({idx}/{len(links)}): {url}")
            try:
                driver.get(url)
                WebDriverWait(driver, 20).until(
                    EC.any_of(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "section.titleContent")),
                        EC.presence_of_element_located((By.CSS_SELECTOR, "div.field_Abstract"))
                    )
                )
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'html.parser')

                # Extract topic, time, abstract
                topic = get_text(soup, "p.favoriteItem")
                date_time = get_text(soup, 'span.defaultTZ')
                abstract = get_text(soup, 'section.field_Abstract')

                # ---------------------------
                # Robust author extraction
                # ---------------------------
                presenting_author = ""  # keep same key & value shape as before
                authors = []            # keep same key & value shape as before (list[str])
                authors_structured = [] # new: detailed list with designation (non-breaking addition)

                person_list = soup.select_one(".PersonList")
                if person_list:
                    # Build structured list with designation
                    for sec in person_list.find_all("section", recursive=False):
                        h = sec.find("h5")
                        if not h:
                            continue
                        head = h.get_text(strip=True)
                        if head.lower().startswith("presenting"):
                            designation = "Presenting Author"
                        else:
                            designation = "Author"

                        for li in sec.select("li.RoleListItem"):
                            name_tag = li.select_one("a")
                            # prefer the affiliation <li> text inside roleAffiliation if present
                            affil_tag = li.select_one("span.roleAffiliation li") or li.select_one("span.roleAffiliation")
                            if not name_tag:
                                continue
                            name = name_tag.get_text(strip=True)
                            affil = affil_tag.get_text(strip=True) if affil_tag else ""
                            authors_structured.append({
                                "name": name,
                                "affiliation": affil,
                                "designation": designation
                            })

                    # Derive presenting_author (string) and authors (list[str]) from structured if possible
                    pa = next((a for a in authors_structured if a["designation"] == "Presenting Author"), None)
                    if pa:
                        presenting_author = f'{pa["name"]} | {pa["affiliation"]}'.strip()
                    # Collect normal authors (exclude presenting if duplicated in "Authors" section)
                    authors = [
                        f'{a["name"]} | {a["affiliation"]}'.strip()
                        for a in authors_structured
                        if a["designation"] == "Author"
                        and f'{a["name"]} | {a["affiliation"]}'.strip() != presenting_author
                    ]

                # Fallbacks if sections are missing or empty
                if not authors_structured:
                    # Fallback: original li.RoleListItem scrape
                    all_authors_elements = soup.select('li.RoleListItem')
                    temp_list = []
                    for author_el in all_authors_elements:
                        name_tag = author_el.select_one('a')
                        affil_tag = author_el.select_one('span.roleAffiliation')
                        if name_tag:
                            name = name_tag.get_text(strip=True)
                            affil = affil_tag.get_text(strip=True) if affil_tag else ""
                            temp_list.append({"name": name, "affiliation": affil, "designation": "Author"})
                    authors_structured = temp_list

                if not presenting_author:
                    # Fallback: original presenter grab
                    presenting_author_name = get_text(soup, 'a.presenter')
                    # Try to get affiliation nearest to presenter; if not, first roleAffiliation
                    presenter_a = soup.select_one('a.presenter')
                    presenter_affil = ""
                    if presenter_a:
                        li_parent = presenter_a.find_parent('li', class_='RoleListItem')
                        if li_parent:
                            affil_li = li_parent.select_one('span.roleAffiliation li') or li_parent.select_one('span.roleAffiliation')
                            presenter_affil = affil_li.get_text(strip=True) if affil_li else ""
                    if not presenter_affil:
                        presenter_affil = get_text(soup, 'span.roleAffiliation')
                    presenting_author = f"{presenting_author_name} | {presenter_affil}".strip()

                if not authors:
                    # Build authors list from structured (excluding presenting)
                    authors = [
                        f'{a["name"]} | {a["affiliation"]}'.strip()
                        for a in authors_structured
                        if f'{a["name"]} | {a["affiliation"]}'.strip() != presenting_author
                    ]

                data = {
                    "url": url,
                    "topic": topic,
                    "date_time": date_time,
                    "abstract": abstract,
                    "presenting_author": presenting_author,  # unchanged shape
                    "authors": authors,                      # unchanged shape
                    # non-breaking addition with designations:
                    "authors_structured": authors_structured
                }

                all_data.append(data)
                print("✅ Extracted successfully.")
                print("📝 Abstract Preview:")
                print("\n".join((data["abstract"] or "").splitlines()[:2]))

            except Exception as e:
                print(f"❌ Failed to extract from {url}\n   Error: {e}")

    finally:
        driver.quit()

    # Save JSON
    output_dir = os.path.dirname(output_file)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(all_data, f, ensure_ascii=False, indent=2)

    print(f"\n✅ All data saved to {output_file}")


# --- Main Execution ---
if __name__ == "__main__":
    json_files = glob.glob("presentation_links_by_session_*.json")
    for json_file in json_files:
        program_id = json_file.split("_")[-1].replace(".json", "")
        output_file = f"aiche_papers_{program_id}.json"
        links = load_all_links_from_json(json_file)
        extract_and_save_presentation_data(links, output_file)


✅ Loaded 2 total presentation links from presentation_links_by_session_3330.json.

🔍 Processing (1/2): https://aiche.confex.com/aiche/2023/meetingapp.cgi/Paper/663122
✅ Extracted successfully.
📝 Abstract Preview:
AbstractIn this talk we will present our recent studies of how the local structure of Zr-sites in zirconia-based catalysts affect their activity and selectivity for reactions important in the upgrading of biomass-derived molecules. Zr site structures were varied using single crystal surfaces, ZrO2particles infiltrated into high surface area supports, and ultra-thin ZrO2films on oxide supports. The latter involved the use of atomic layer deposition (ALD) to produce conformal ZrO2films less than 2 nm in thickness, as well as isolated Zr sites. Specific examples that will be presented include structure-activity relationships for the dehydra-decylcization of cyclic ethers, transfer hydrogenation and etherification of hydroxymethylfurfural, and Diels–Alder cycloaddition of ethylene

In [ ]:
!pip install selenium
!pip install webdriver-manager
!apt-get update
!apt-get install -y chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
import sys
sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')
import requests
from bs4 import BeautifulSoup

# URL of one of the session pages
session_url = "https://aiche.confex.com/aiche/2023/meetingapp.cgi/Session/54130"

# Download the page content
response = requests.get(session_url)
soup = BeautifulSoup(response.text, "html.parser")

# Print the HTML to inspect it
print(soup.prettify())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.2/499.2 kB 26.9 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,931 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy In

In [ ]:
import os
import re
import json
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from google.colab import drive # Import google.colab.drive

# Mount Google Drive
drive.mount('/content/drive')
# Define the base directory for saving files in Google Drive
DRIVE_SAVE_DIR = "/content/drive/My Drive/UGP"
# Ensure the directory exists
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)


PROGRAM_URLS_FILE = "2024_links_remain.json" # Keep this path local, or move if preferred
# Setting BASE_URL back to 2024 as requested
BASE_URL = "https://aiche.confex.com/aiche/2024/meetingapp.cgi/"

# ----------------------------
# Stage 1: scrape sessions from a Program page
# ----------------------------
def scrape_program_page_sessions(program_url: str):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--remote-debugging-pipe")
    driver = webdriver.Chrome(options=options)
    try:
        driver.get(program_url)
        WebDriverWait(driver, 60).until(
            EC.presence_of_element_located((By.CLASS_NAME, "CalendarList"))
        )
        soup = BeautifulSoup(driver.page_source, "html.parser")
        ul_calendar = soup.find("ul", class_="Calendar")
        page_data = []
        if ul_calendar:
            calendar_lists = ul_calendar.find_all("ul", class_="CalendarList")
            for cal_list in calendar_lists:
                date_li = cal_list.find("li")
                date = None
                if date_li:
                    date_span = date_li.find("span", class_="defaultTZ")
                    if date_span:
                        date = date_span.get_text(strip=True)

                time_tag = cal_list.find("time", class_="first")
                time_range = time_tag.get_text(strip=True) if time_tag else None

                sessions = []
                session_sections = cal_list.find_all("section", class_="itemCalendar")
                for session in session_sections:
                    a_tag = session.find("a")
                    session_title = a_tag.get_text(strip=True) if a_tag else None
                    session_link = a_tag['href'] if a_tag and 'href' in a_tag.attrs else None

                    speakers = []
                    top_display = session.find("span", class_="topDisplay")
                    if top_display:
                        bolds = top_display.find_all("b")
                        for b in bolds:
                            speakers.append(b.get_text(strip=True))

                    location = None
                    prop_info = session.find("ul", class_="propertyInfo")
                    if prop_info:
                        prop_name = prop_info.find("li", class_="propertyName")
                        if prop_name:
                            location = prop_name.get_text(strip=True)

                    sessions.append({
                        "title": session_title,
                        "link": session_link,
                        "speakers": speakers,
                        "location": location
                    })

                page_data.append({
                    "date": date,
                    "time": time_range,
                    "sessions": sessions
                })
        else:
            print(f"❌ <ul class='Calendar'> not found for {program_url}.")
        return page_data
    finally:
        driver.quit()

# ----------------------------
# Stage 2: collect paper links from a Session page
# ----------------------------
def get_all_session_urls(json_file_path):
    with open(json_file_path, "r", encoding="utf-8") as file:
        sessions_data = json.load(file)
    session_urls = []
    for entry in sessions_data:
        for session in entry.get("sessions", []):
            session_link = session.get("link")
            if session_link:
                session_urls.append(BASE_URL + session_link)
    return session_urls

def extract_presentation_links_from_live_page(url):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--remote-debugging-pipe")
    driver = None
    try:
        driver = webdriver.Chrome(options=options)
        print(f"🔗 Opening URL: {url}")
        driver.get(url)

        WebDriverWait(driver, 40).until_not(
            EC.text_to_be_present_in_element(
                (By.TAG_NAME, "body"),
                "please wait while the program loads"
            )
        )
        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "section.field_ChildList_PaperSlot"))
        )
        time.sleep(5)
        print("✅ Presentations section found.")
        print("⏳ Waiting 10 seconds for dynamic content inside <ul> to load...")
        time.sleep(10)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "ul.PaperSlot a[href^='Paper/']"))
        )
        print("✅ At least one paper link found inside <ul.PaperSlot>.")

        soup = BeautifulSoup(driver.page_source, 'html.parser')
        presentation_section = soup.select_one("section.field_ChildList_PaperSlot")
        if not presentation_section:
            print("❌ Presentations section not found after waiting.")
            return []

        presentation_section_ul = presentation_section.find("ul", class_="PaperSlot")
        if not presentation_section_ul:
            print("❌ <ul> with class PaperSlot not found.")
            return []

        links = []
        for a_tag in presentation_section_ul.find_all("a", href=True):
            href = a_tag["href"]
            if href.startswith("Paper/"):
                full_link = BASE_URL + href
                links.append(full_link)
        return links
    except Exception as e:
        print(f"❌ Error: {e}")
        return []
    finally:
        if driver:
            driver.quit()

# ----------------------------
# Stage 3: scrape paper details from a Paper page
# ----------------------------
def get_text(soup, selector):
    elements = soup.select(selector)
    if not elements:
        return "Not found"
    return " | ".join([el.get_text(strip=True) for el in elements])

def load_all_links(json_file_path):
    if not os.path.exists(json_file_path):
        print(f"❌ File {json_file_path} not found.")
        return []
    with open(json_file_path, "r", encoding="utf-8") as f:
        session_links = json.load(f)
    all_links = []
    for _, links in session_links.items():
        all_links.extend(links)
    print(f"✅ Loaded {len(all_links)} total presentation links.")
    return all_links

def extract_and_save_presentation_data(links, output_file, existing_papers):
    if not links:
        print("⚠️ No links to process.")
        return existing_papers # Return existing data if no new links

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--remote-debugging-pipe")
    driver = webdriver.Chrome(options=options)

    # Start with existing data
    all_data = existing_papers

    try:
        for idx, url in enumerate(links, start=1):
            print(f"\n🔍 Processing ({idx}/{len(links)}): {url}")
            try:
                driver.get(url)
                WebDriverWait(driver, 20).until(
                    EC.any_of(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "section.titleContent")),
                        EC.presence_of_element_located((By.CSS_SELECTOR, "div.field_Abstract"))
                    )
                )
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'html.parser')

                topic = get_text(soup, "p.favoriteItem")
                date_time = get_text(soup, 'span.defaultTZ')
                abstract = get_text(soup, 'section.field_Abstract')

                presenting_author = ""
                authors = []
                authors_structured = []

                person_list = soup.select_one(".PersonList")
                if person_list:
                    for sec in person_list.find_all("section", recursive=False):
                        h = sec.find("h5")
                        if not h:
                            continue
                        head = h.get_text(strip=True)
                        if head.lower().startswith("presenting"):
                            designation = "Presenting Author"
                        else:
                            designation = "Author"

                        for li in sec.select("li.RoleListItem"):
                            name_tag = li.select_one("a")
                            affil_tag = li.select_one("span.roleAffiliation li") or li.select_one("span.roleAffiliation")
                            if not name_tag:
                                continue
                            name = name_tag.get_text(strip=True)
                            affil = affil_tag.get_text(strip=True) if affil_tag else ""
                            authors_structured.append({
                                "name": name,
                                "affiliation": affil,
                                "designation": designation
                            })

                    pa = next((a for a in authors_structured if a["designation"] == "Presenting Author"), None)
                    if pa:
                        presenting_author = f'{pa["name"]} | {pa["affiliation"]}'.strip()

                    authors = [
                        f'{a["name"]} | {a["affiliation"]}'.strip()
                        for a in authors_structured
                        if a["designation"] == "Author"
                        and f'{a["name"]} | {a["affiliation"]}'.strip() != presenting_author
                    ]

                if not authors_structured:
                    all_authors_elements = soup.select('li.RoleListItem')
                    temp_list = []
                    for author_el in all_authors_elements:
                        name_tag = author_el.select_one('a')
                        affil_tag = author_el.select_one('span.roleAffiliation')
                        if name_tag:
                            name = name_tag.get_text(strip=True)
                            affil = affil_tag.get_text(strip=True) if affil_tag else ""
                            temp_list.append({"name": name, "affiliation": affil, "designation": "Author"})
                    authors_structured = temp_list

                if not presenting_author:
                    presenting_author_name = get_text(soup, 'a.presenter')
                    presenter_a = soup.select_one('a.presenter')
                    presenter_affil = ""
                    if presenter_a:
                        li_parent = presenter_a.find_parent('li', class_='RoleListItem')
                        if li_parent:
                            affil_li = li_parent.select_one('span.roleAffiliation li') or li_parent.select_one('span.roleAffiliation')
                            presenter_affil = affil_li.get_text(strip=True) if affil_li else ""
                             # Try to get affiliation nearest to presenter; if not, first roleAffiliation
                    if not presenter_affil:
                        presenter_affil = get_text(soup, 'span.roleAffiliation')
                    presenting_author = f"{presenting_author_name} | {presenter_affil}".strip()

                if not authors:
                    authors = [
                        f'{a["name"]} | {a["affiliation"]}'.strip()
                        for a in authors_structured
                        if f'{a["name"]} | {a["affiliation"]}'.strip() != presenting_author
                    ]

                data = {
                    "url": url,
                    "topic": topic,
                    "date_time": date_time,
                    "abstract": abstract,
                    "presenting_author": presenting_author,
                    "authors": authors,
                    "authors_structured": authors_structured
                }

                all_data.append(data)
                print("✅ Extracted successfully.")
                print("📝 Abstract Preview:")
                print("\n".join((data["abstract"] or "").splitlines()[:2]))

                # Removed periodic save here


            except Exception as e:
                print(f"❌ Failed to extract from {url}\n   Error: {e}")

    finally:
        driver.quit()

    # Final save at the end of processing remaining links for this program ID
    output_dir = os.path.dirname(output_file)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(all_data, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Final Stage 3 save -> {output_file}")

    return all_data # Return the updated data


# ----------------------------
# Helpers
# ----------------------------
def read_program_urls():
    # Read the program URLs file from Google Drive
    drive_program_urls_path = os.path.join(DRIVE_SAVE_DIR, PROGRAM_URLS_FILE)
    if not os.path.exists(drive_program_urls_path):
         # Fallback to local if not in Drive (e.g., first run before saving to Drive)
         print(f"⚠️ {PROGRAM_URLS_FILE} not found in Google Drive. Checking local.")
         if not os.path.exists(PROGRAM_URLS_FILE):
             raise FileNotFoundError(f"{PROGRAM_URLS_FILE} not found locally or in Google Drive.")
         file_path = PROGRAM_URLS_FILE
    else:
        file_path = drive_program_urls_path

    with open(file_path, "r", encoding="utf-8") as f:
        payload = json.load(f)
    if isinstance(payload, dict) and "urls" in payload:
        return payload["urls"]
    if isinstance(payload, list):
        return payload
    # Assuming the local file was "2024_links_remain.json" and needs to be moved/handled
    # For simplicity, let's assume the user will place it in UGP or handle it manually
    raise ValueError(f"Invalid format in {file_path}. Expected a list or object with key 'urls'.")


def program_id_from_url(url: str) -> str:
    m = re.search(r"/Program/(\d+)", url)
    return m.group(1) if m else re.sub(r"\W+", "_", url.strip("/"))

# ----------------------------
# Orchestrator
# ----------------------------
def main():
    program_urls = read_program_urls()
    print(f"Found {len(program_urls)} Program URLs.")

    for idx, program_url in enumerate(program_urls, start=1):
        pid = program_id_from_url(program_url)
        # Update file paths to save to Google Drive
        sessions_file = os.path.join(DRIVE_SAVE_DIR, f"sessions_{pid}.json")
        links_file = os.path.join(DRIVE_SAVE_DIR, f"presentation_links_{pid}.json")
        papers_file = os.path.join(DRIVE_SAVE_DIR, f"aiche_papers_{pid}.json")


        print(f"\n====================")
        print(f"({idx}/{len(program_urls)}) Program: {program_url} -> ID: {pid}")
        print(f"Output files will be saved to: {DRIVE_SAVE_DIR}") # Updated print message
        print(f"====================")

        # Stage 1: Scraping sessions (always run to ensure sessions_file is up-to-date)
        print("[Stage 1] Scraping sessions...")
        # Check if sessions_file exists to skip scraping if already done
        if os.path.exists(sessions_file):
            try:
                with open(sessions_file, "r", encoding="utf-8") as f:
                    sessions_data = json.load(f)
                print(f"✅ Stage 1 file found: {sessions_file}. Skipping Stage 1 for this program.")
            except json.JSONDecodeError:
                print(f"❌ Error decoding {sessions_file}. Proceeding to re-scrape session URLs.")
                sessions_data = scrape_program_page_sessions(program_url)
                with open(sessions_file, "w", encoding="utf-8") as f:
                    json.dump(sessions_data, f, indent=2, ensure_ascii=False)
                print(f"✅ Stage 1 saved -> {sessions_file}")
        else:
            sessions_data = scrape_program_page_sessions(program_url)
            with open(sessions_file, "w", encoding="utf-8") as f:
                json.dump(sessions_data, f, indent=2, ensure_ascii=False)
            print(f"✅ Stage 1 saved -> {sessions_file}")


        # Stage 2: Building presentation links (skip if links_file exists)
        print("[Stage 2] Building presentation links...")
        links_file_exists = os.path.exists(links_file)

        if links_file_exists:
            try:
                with open(links_file, "r", encoding="utf-8") as f:
                    all_links_by_session = json.load(f)
                print(f"✅ Stage 2 file found: {links_file}. Skipping Stage 2 for this program.")
                # No need to process session_urls if skipping Stage 2 based on file existence
                # all_links_by_session is loaded and will be used for Stage 3 if not skipped
            except json.JSONDecodeError:
                print(f"❌ Error decoding {links_file}. Cannot skip Stage 2, proceeding to re-scrape.")
                links_file_exists = False # Force re-scraping


        if not links_file_exists:
            session_urls = get_all_session_urls(sessions_file) # Need session URLs to process Stage 2
            all_links_by_session = {}
            # Load existing links if the file exists (for resuming if not skipping the whole stage)
            if os.path.exists(links_file): # This check is redundant if links_file_exists is False, but kept for clarity/original flow
                try:
                    with open(links_file, "r", encoding="utf-8") as f:
                        loaded_links = json.load(f)
                        all_links_by_session.update(loaded_links) # Merge existing links
                    print(f"Loaded existing presentation links from {links_file} for partial resume.")
                except json.JSONDecodeError:
                     print(f"Error decoding {links_file}. Starting Stage 2 from scratch.")
                     all_links_by_session = {}


            processed_sessions = set(all_links_by_session.keys())
            sessions_to_process = [url for url in session_urls if url not in processed_sessions]

            if not sessions_to_process and os.path.exists(links_file): # Check if all sessions processed AND file exists
                 print("✅ All session links already processed for this program based on existing file.")
            else:
                print(f"ℹ️ Processing {len(sessions_to_process)} / {len(session_urls)} sessions.")
                # Re-initialize driver for Stage 2
                options = Options()
                options.add_argument("--headless")
                options.add_argument("--no-sandbox")
                options.add_argument("--remote-debugging-pipe")
                driver = webdriver.Chrome(options=options)
                try:
                    for sidx, session_url in enumerate(sessions_to_process, start=1):
                        print(f"   - Processing Session {sidx}/{len(sessions_to_process)}: {session_url}")
                        links = extract_presentation_links_from_live_page(session_url)
                        all_links_by_session[session_url] = links
                        print(f"     -> {len(links)} links")
                        # Removed periodic save here


                finally:
                     driver.quit()

            # Final save at the end of Stage 2 if it wasn't skipped
            if not links_file_exists:
                 with open(links_file, "w", encoding="utf-8") as f:
                     json.dump(all_links_by_session, f, indent=2)
                 print(f"✅ Stage 2 saved -> {links_file}")


        # Stage 3: Scraping paper details (skip if papers_file exists)
        print("[Stage 3] Scraping paper details...")
        papers_file_exists = os.path.exists(papers_file)

        if papers_file_exists:
            try:
                # Attempt to load the file to confirm it's valid JSON and truly completed
                with open(papers_file, "r", encoding="utf-8") as f:
                     existing_papers = json.load(f)
                print(f"✅ Stage 3 file found and loaded: {papers_file}. Skipping Stage 3 for this program.")
                # If the file exists and is valid, we can skip the rest of Stage 3 for this program ID
                continue # Move to the next program_url in the main loop

            except json.JSONDecodeError:
                 print(f"❌ Error decoding {papers_file}. File might be incomplete. Proceeding to re-scrape/resume Stage 3.")
                 # If decoding fails, we assume the file is incomplete and proceed with scraping/resuming
                 papers_file_exists = False # Ensure we don't skip based on the failed load


        # If Stage 3 wasn't skipped (either file didn't exist or decode failed)
        # Load links from Stage 2 output (either newly scraped or loaded from file)
        # Ensure all_links_by_session is available, load from file if Stage 2 was skipped but Stage 3 wasn't
        if 'all_links_by_session' not in locals() and os.path.exists(links_file):
             try:
                 with open(links_file, "r", encoding="utf-8") as f:
                     all_links_by_session = json.load(f)
                 print(f"Loaded Stage 2 links from {links_file} for Stage 3 processing.")
             except json.JSONDecodeError:
                  print(f"❌ Error decoding {links_file}. Cannot load Stage 2 links for Stage 3.")
                  all_links_by_session = {} # Initialize empty to avoid error

        links = load_all_links(links_file) # This function needs the path to the links file

        # Load existing paper data if the file exists (for resuming if not skipping the whole stage)
        existing_papers = [] # Initialize empty for scraping
        if os.path.exists(papers_file): # This check is now primarily for loading data to resume within the stage if needed
             try:
                 with open(papers_file, "r", encoding="utf-8") as f:
                     existing_papers = json.load(f)
                 print(f"Loaded existing paper data from {papers_file} for partial resume.")
             except json.JSONDecodeError:
                 print(f"Error decoding {papers_file}. Starting Stage 3 from scratch.")
                 existing_papers = []


        processed_paper_urls = {paper["url"] for paper in existing_papers}
        paper_links_to_process = [link for link in links if link not in processed_paper_urls]

        if not paper_links_to_process and os.path.exists(papers_file): # Check if all papers processed AND file exists
             print("✅ All paper links already processed for this program based on existing file.")
             # Since all papers are processed and the file exists, we can explicitly move to the next program URL
             continue # Move to the next program_url in the main loop

        else: # If there are papers to process or the file didn't exist/decode failed
             print(f"ℹ️ Processing {len(paper_links_to_process)} / {len(links)} papers.")
             # Need to load existing data first, then append new data - Handled by 'all_data = existing_papers' below
             all_data = existing_papers # Start with already processed data
             options = Options() # Re-initialize driver options
             options.add_argument("--headless")
             options.add_argument("--no-sandbox")
             options.add_argument("--remote-debugging-pipe")
             driver = webdriver.Chrome(options=options)

             try:
                 for idx, url in enumerate(paper_links_to_process, start=1):
                     print(f"\n🔍 Processing ({idx}/{len(paper_links_to_process)}): {url}")
                     try:
                         driver.get(url)
                         WebDriverWait(driver, 20).until(
                             EC.any_of(
                                 EC.presence_of_element_located((By.CSS_SELECTOR, "section.titleContent")),
                                 EC.presence_of_element_located((By.CSS_SELECTOR, "div.field_Abstract"))
                             )
                         )
                         time.sleep(2)
                         soup = BeautifulSoup(driver.page_source, 'html.parser')

                         topic = get_text(soup, "p.favoriteItem")
                         date_time = get_text(soup, 'span.defaultTZ')
                         abstract = get_text(soup, 'section.field_Abstract')

                         presenting_author = ""
                         authors = []
                         authors_structured = []

                         person_list = soup.select_one(".PersonList")
                         if person_list:
                             for sec in person_list.find_all("section", recursive=False):
                                 h = sec.find("h5")
                                 if not h:
                                     continue
                                 head = h.get_text(strip=True)
                                 if head.lower().startswith("presenting"):
                                     designation = "Presenting Author"
                                 else:
                                     designation = "Author"

                                 for li in sec.select("li.RoleListItem"):
                                     name_tag = li.select_one("a")
                                     affil_tag = li.select_one("span.roleAffiliation li") or li.select_one("span.roleAffiliation")
                                     if not name_tag:
                                         continue
                                     name = name_tag.get_text(strip=True)
                                     affil = affil_tag.get_text(strip=True) if affil_tag else ""
                                     authors_structured.append({
                                         "name": name,
                                         "affiliation": affil,
                                         "designation": designation
                                     })

                             pa = next((a for a in authors_structured if a["designation"] == "Presenting Author"), None)
                             if pa:
                                 presenting_author = f'{pa["name"]} | {pa["affiliation"]}'.strip()

                             authors = [
                                 f'{a["name"]} | {a["affiliation"]}'.strip()
                                 for a in authors_structured
                                 if a["designation"] == "Author"
                                 and f'{a["name"]} | {a["affiliation"]}'.strip() != presenting_author
                             ]

                         if not authors_structured:
                             all_authors_elements = soup.select('li.RoleListItem')
                             temp_list = []
                             for author_el in all_authors_elements:
                                 name_tag = author_el.select_one('a')
                                 affil_tag = author_el.select_one('span.roleAffiliation')
                                 if name_tag:
                                     name = name_tag.get_text(strip=True)
                                     affil = affil_tag.get_text(strip=True) if affil_tag else ""
                                     temp_list.append({"name": name, "affiliation": affil, "designation": "Author"})
                             authors_structured = temp_list

                         if not presenting_author:
                             presenting_author_name = get_text(soup, 'a.presenter')
                             presenter_a = soup.select_one('a.presenter')
                             presenter_affil = ""
                             if presenter_a:
                                 li_parent = presenter_a.find_parent('li', class_='RoleListItem')
                                 if li_parent:
                                     affil_li = li_parent.select_one('span.roleAffiliation li') or li_parent.select_one('span.roleAffiliation')
                                     presenter_affil = affil_li.get_text(strip=True) if affil_li else ""
                             if not presenter_affil:
                                 presenter_affil = get_text(soup, 'span.roleAffiliation')
                             presenting_author = f"{presenting_author_name} | {presenter_affil}".strip()

                         if not authors:
                             authors = [
                                 f'{a["name"]} | {a["affiliation"]}'.strip()
                                 for a in authors_structured
                                 if f'{a["name"]} | {a["affiliation"]}'.strip() != presenting_author
                             ]

                         data = {
                             "url": url,
                             "topic": topic,
                             "date_time": date_time,
                             "abstract": abstract,
                             "presenting_author": presenting_author,
                             "authors": authors,
                             "authors_structured": authors_structured
                         }

                         all_data.append(data)
                         print("✅ Extracted successfully.")
                         print("📝 Abstract Preview:")
                         print("\n".join((data["abstract"] or "").splitlines()[:2]))

                         # Removed periodic save here


                     except Exception as e:
                         print(f"❌ Failed to extract from {url}\n   Error: {e}")

             finally:
                 driver.quit()

             # Final save at the end of processing remaining links for this program ID
             output_dir = os.path.dirname(papers_file)
             if output_dir:
                 os.makedirs(output_dir, exist_ok=True)

             with open(papers_file, 'w', encoding='utf-8') as f:
                 json.dump(all_data, f, ensure_ascii=False, indent=2)

             print(f"\n✅ Final Stage 3 save -> {papers_file}")


    print("\n🎉 Done for all Program URLs.")

if __name__ == "__main__":
    main()

Streaming output truncated to the last 5000 lines.
🔍 Processing (22/101): https://aiche.confex.com/aiche/2024/meetingapp.cgi/Paper/701372
✅ Extracted successfully.
📝 Abstract Preview:
Not found

🔍 Processing (23/101): https://aiche.confex.com/aiche/2024/meetingapp.cgi/Paper/691928
✅ Extracted successfully.
📝 Abstract Preview:
AbstractIn the next two or three decades most of today’s large-scale chemical and fuel processes will need to be replaced by low-greenhouse alternatives. In most cases the new processes will be more expensive than the incumbent high-emitting processes. The only way this can happen if government policy favors introduction of the new process. But if government policy favors an expensive or ineffective new process, society might pay a high cost for little benefit. Clearly it would be beneficial if the future policies are based on an accurate understanding about which new processes are most beneficial to society; in an ideal world this accurate understanding would com

In [ ]:
import os
import re
import json
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from google.colab import drive # Import google.colab.drive

# Mount Google Drive
drive.mount('/content/drive')
# Define the base directory for saving files in Google Drive
DRIVE_SAVE_DIR = "/content/drive/My Drive/UGP"
# Ensure the directory exists
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)


PROGRAM_URLS_FILE = "2024_links_remain.json" # Keep this path local, or move if preferred
# Setting BASE_URL back to 2024 as requested
BASE_URL = "https://aiche.confex.com/aiche/2024/meetingapp.cgi/"

# ----------------------------
# Stage 1: scrape sessions from a Program page
# ----------------------------
def scrape_program_page_sessions(program_url: str):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--remote-debugging-pipe")
    # Added this option to help with WebDriverException in Colab
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=options)
    try:
        driver.get(program_url)
        WebDriverWait(driver, 60).until(
            EC.presence_of_element_located((By.CLASS_NAME, "CalendarList"))
        )
        soup = BeautifulSoup(driver.page_source, "html.parser")
        ul_calendar = soup.find("ul", class_="Calendar")
        page_data = []
        if ul_calendar:
            calendar_lists = ul_calendar.find_all("ul", class_="CalendarList")
            for cal_list in calendar_lists:
                date_li = cal_list.find("li")
                date = None
                if date_li:
                    date_span = date_li.find("span", class_="defaultTZ")
                    if date_span:
                        date = date_span.get_text(strip=True)

                time_tag = cal_list.find("time", class_="first")
                time_range = time_tag.get_text(strip=True) if time_tag else None

                sessions = []
                session_sections = cal_list.find_all("section", class_="itemCalendar")
                for session in session_sections:
                    a_tag = session.find("a")
                    session_title = a_tag.get_text(strip=True) if a_tag else None
                    session_link = a_tag['href'] if a_tag and 'href' in a_tag.attrs else None

                    speakers = []
                    top_display = session.find("span", class_="topDisplay")
                    if top_display:
                        bolds = top_display.find_all("b")
                        for b in bolds:
                            speakers.append(b.get_text(strip=True))

                    location = None
                    prop_info = session.find("ul", class_="propertyInfo")
                    if prop_info:
                        prop_name = prop_info.find("li", class_="propertyName")
                        if prop_name:
                            location = prop_name.get_text(strip=True)

                    sessions.append({
                        "title": session_title,
                        "link": session_link,
                        "speakers": speakers,
                        "location": location
                    })

                page_data.append({
                    "date": date,
                    "time": time_range,
                    "sessions": sessions
                })
        else:
            print(f"❌ <ul class='Calendar'> not found for {program_url}.")
        return page_data
    finally:
        driver.quit()

# ----------------------------
# Stage 2: collect paper links from a Session page
# ----------------------------
def get_all_session_urls(json_file_path):
    with open(json_file_path, "r", encoding="utf-8") as file:
        sessions_data = json.load(file)
    session_urls = []
    for entry in sessions_data:
        for session in entry.get("sessions", []):
            session_link = session.get("link")
            if session_link:
                session_urls.append(BASE_URL + session_link)
    return session_urls

# Modified to accept and reuse a driver instance
def extract_presentation_links_from_live_page(driver, url):
    try:
        print(f"🔗 Opening URL: {url}")
        driver.get(url)

        WebDriverWait(driver, 40).until_not(
            EC.text_to_be_present_in_element(
                (By.TAG_NAME, "body"),
                "please wait while the program loads"
            )
        )
        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "section.field_ChildList_PaperSlot"))
        )
        time.sleep(5)
        print("✅ Presentations section found.")
        print("⏳ Waiting 10 seconds for dynamic content inside <ul> to load...")
        time.sleep(10)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "ul.PaperSlot a[href^='Paper/']"))
        )
        print("✅ At least one paper link found inside <ul.PaperSlot>.")

        soup = BeautifulSoup(driver.page_source, 'html.parser')
        presentation_section = soup.select_one("section.field_ChildList_PaperSlot")
        if not presentation_section:
            print("❌ Presentations section not found after waiting.")
            return []

        presentation_section_ul = presentation_section.find("ul", class_="PaperSlot")
        if not presentation_section_ul:
            print("❌ <ul> with class PaperSlot not found.")
            return []

        links = []
        for a_tag in presentation_section_ul.find_all("a", href=True):
            href = a_tag["href"]
            if href.startswith("Paper/"):
                full_link = BASE_URL + href
                links.append(full_link)
        return links
    except Exception as e:
        print(f"❌ Error: {e}")
        return []
    finally:
        # Note: The driver will be closed outside this function, in the main loop
        pass # Ensure the finally block is not empty


# ----------------------------
# Stage 3: scrape paper details from a Paper page
# ----------------------------
def get_text(soup, selector):
    elements = soup.select(selector)
    if not elements:
        return "Not found"
    return " | ".join([el.get_text(strip=True) for el in elements])

def load_all_links(json_file_path):
    if not os.path.exists(json_file_path):
        print(f"❌ File {json_file_path} not found.")
        return []
    with open(json_file_path, "r", encoding="utf-8") as f:
        session_links = json.load(f)
    all_links = []
    for _, links in session_links.items():
        all_links.extend(links)
    print(f"✅ Loaded {len(all_links)} total presentation links.")
    return all_links

# Modified to accept and reuse a driver instance
def extract_and_save_presentation_data(driver, links, output_file, existing_papers):
    if not links:
        print("⚠️ No links to process.")
        return existing_papers # Return existing data if no new links


    # Start with existing data
    all_data = existing_papers

    try:
        for idx, url in enumerate(links, start=1):
            print(f"\n🔍 Processing ({idx}/{len(links)}): {url}")
            try:
                driver.get(url)
                WebDriverWait(driver, 20).until(
                    EC.any_of(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "section.titleContent")),
                        EC.presence_of_element_located((By.CSS_SELECTOR, "div.field_Abstract"))
                    )
                )
                time.sleep(2)
                soup = BeautifulSoup(driver.page_source, 'html.parser')

                topic = get_text(soup, "p.favoriteItem")
                date_time = get_text(soup, 'span.defaultTZ')
                abstract = get_text(soup, 'section.field_Abstract')

                presenting_author = ""
                authors = []
                authors_structured = []

                person_list = soup.select_one(".PersonList")
                if person_list:
                    for sec in person_list.find_all("section", recursive=False):
                        h = sec.find("h5")
                        if not h:
                            continue
                        head = h.get_text(strip=True)
                        if head.lower().startswith("presenting"):
                            designation = "Presenting Author"
                        else:
                            designation = "Author"

                        for li in sec.select("li.RoleListItem"):
                            name_tag = li.select_one("a")
                            affil_tag = li.select_one('span.roleAffiliation li') or li.select_one("span.roleAffiliation")
                            if not name_tag:
                                continue
                            name = name_tag.get_text(strip=True)
                            affil = affil_tag.get_text(strip=True) if affil_tag else ""
                            authors_structured.append({
                                "name": name,
                                "affiliation": affil,
                                "designation": designation
                            })

                    pa = next((a for a in authors_structured if a["designation"] == "Presenting Author"), None)
                    if pa:
                        presenting_author = f'{pa["name"]} | {pa["affiliation"]}'.strip()

                    authors = [
                        f'{a["name"]} | {a["affiliation"]}'.strip()
                        for a in authors_structured
                        if a["designation"] == "Author"
                        and f'{a["name"]} | {a["affiliation"]}'.strip() != presenting_author
                    ]

                if not authors_structured:
                    all_authors_elements = soup.select('li.RoleListItem')
                    temp_list = []
                    for author_el in all_authors_elements:
                        name_tag = author_el.select_one('a')
                        affil_tag = author_el.select_one('span.roleAffiliation')
                        if name_tag:
                            name = name_tag.get_text(strip=True)
                            affil = affil_tag.get_text(strip=True) if affil_tag else ""
                            temp_list.append({"name": name, "affiliation": affil, "designation": "Author"})
                    authors_structured = temp_list

                if not presenting_author:
                    presenting_author_name = get_text(soup, 'a.presenter')
                    presenter_a = soup.select_one('a.presenter')
                    presenter_affil = ""
                    if presenter_a:
                        li_parent = presenter_a.find_parent('li', class_='RoleListItem')
                        if li_parent:
                            affil_li = li_parent.select_one('span.roleAffiliation li') or li_parent.select_one('span.roleAffiliation')
                            presenter_affil = affil_li.get_text(strip=True) if affil_li else ""
                             # Try to get affiliation nearest to presenter; if not, first roleAffiliation
                    if not presenter_affil:
                        presenter_affil = get_text(soup, 'span.roleAffiliation')
                    presenting_author = f"{presenting_author_name} | {presenter_affil}".strip()

                if not authors:
                    authors = [
                        f'{a["name"]} | {a["affiliation"]}'.strip()
                        for a in authors_structured
                        if f'{a["name"]} | {a["affiliation"]}'.strip() != presenting_author
                    ]

                data = {
                    "url": url,
                    "topic": topic,
                    "date_time": date_time,
                    "abstract": abstract,
                    "presenting_author": presenting_author,
                    "authors": authors,
                    "authors_structured": authors_structured
                }

                all_data.append(data)
                print("✅ Extracted successfully.")
                print("📝 Abstract Preview:")
                print("\n".join((data["abstract"] or "").splitlines()[:2]))


                # Removed periodic save here


            except Exception as e:
                print(f"❌ Failed to extract from {url}\n   Error: {e}")

    finally:
        # This finally block was causing the syntax error.
        # Ensure it has a statement inside.
        pass


    # Final save at the end of processing remaining links for this program ID
    output_dir = os.path.dirname(output_file)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(all_data, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Final Stage 3 save -> {output_file}")

    return all_data # Return the updated data


# ----------------------------
# Helpers
# ----------------------------
def read_program_urls():
    # Read the program URLs file from Google Drive
    drive_program_urls_path = os.path.join(DRIVE_SAVE_DIR, PROGRAM_URLS_FILE)
    if not os.path.exists(drive_program_urls_path):
         # Fallback to local if not in Drive (e.g., first run before saving to Drive)
         print(f"⚠️ {PROGRAM_URLS_FILE} not found in Google Drive. Checking local.")
         if not os.path.exists(PROGRAM_URLS_FILE):
             raise FileNotFoundError(f"{PROGRAM_URLS_FILE} not found locally or in Google Drive.")
         file_path = PROGRAM_URLS_FILE
    else:
        file_path = drive_program_urls_path

    with open(file_path, "r", encoding="utf-8") as f:
        payload = json.load(f)
    if isinstance(payload, dict) and "urls" in payload:
        return payload["urls"]
    if isinstance(payload, list):
        return payload
    # Assuming the local file was "2024_links_remain.json" and needs to be moved/handled
    # For simplicity, let's assume the user will place it in UGP or handle it manually
    raise ValueError(f"Invalid format in {file_path}. Expected a list or object with key 'urls'.")


def program_id_from_url(url: str) -> str:
    m = re.search(r"/Program/(\d+)", url)
    return m.group(1) if m else re.sub(r"\W+", "_", url.strip("/"))

# ----------------------------
# Orchestrator
# ----------------------------
def main():
    program_urls = read_program_urls()
    print(f"Found {len(program_urls)} Program URLs.")

    for idx, program_url in enumerate(program_urls, start=1):
        pid = program_id_from_url(program_url)
        # Update file paths to save to Google Drive
        sessions_file = os.path.join(DRIVE_SAVE_DIR, f"sessions_{pid}.json")
        links_file = os.path.join(DRIVE_SAVE_DIR, f"presentation_links_{pid}.json")
        papers_file = os.path.join(DRIVE_SAVE_DIR, f"aiche_papers_{pid}.json")


        print(f"\n====================")
        print(f"({idx}/{len(program_urls)}) Program: {program_url} -> ID: {pid}")
        print(f"Output files will be saved to: {DRIVE_SAVE_DIR}") # Updated print message
        print(f"====================")

        # Stage 1: Scraping sessions (always run to ensure sessions_file is up-to-date)
        print("[Stage 1] Scraping sessions...")
        # Check if sessions_file exists to skip scraping if already done
        if os.path.exists(sessions_file):
            try:
                with open(sessions_file, "r", encoding="utf-8") as f:
                    sessions_data = json.load(f)
                print(f"✅ Stage 1 file found: {sessions_file}. Skipping Stage 1 for this program.")
            except json.JSONDecodeError:
                print(f"❌ Error decoding {sessions_file}. Proceeding to re-scrape session URLs.")
                sessions_data = scrape_program_page_sessions(program_url)
                with open(sessions_file, "w", encoding="utf-8") as f:
                    json.dump(sessions_data, f, indent=2, ensure_ascii=False)
                print(f"✅ Stage 1 saved -> {sessions_file}")
        else:
            sessions_data = scrape_program_page_sessions(program_url)
            with open(sessions_file, "w", encoding="utf-8") as f:
                json.dump(sessions_data, f, indent=2, ensure_ascii=False)
            print(f"✅ Stage 1 saved -> {sessions_file}")


        # Stage 2: Building presentation links (skip if links_file exists)
        print("[Stage 2] Building presentation links...")
        links_file_exists = os.path.exists(links_file)

        if links_file_exists:
            try:
                with open(links_file, "r", encoding="utf-8") as f:
                    all_links_by_session = json.load(f)
                print(f"✅ Stage 2 file found: {links_file}. Skipping Stage 2 for this program.")
                # No need to process session_urls if skipping Stage 2 based on file existence
                # all_links_by_session is loaded and will be used for Stage 3 if not skipped
            except json.JSONDecodeError:
                print(f"❌ Error decoding {links_file}. Cannot skip Stage 2, proceeding to re-scrape.")
                links_file_exists = False # Force re-scraping


        if not links_file_exists:
            session_urls = get_all_session_urls(sessions_file) # Need session URLs to process Stage 2
            all_links_by_session = {}
            # Load existing links if the file exists (for resuming if not skipping the whole stage)
            if os.path.exists(links_file): # This check is redundant if links_file_exists is False, but kept for clarity/original flow
                try:
                    with open(links_file, "r", encoding="utf-8") as f:
                        loaded_links = json.load(f)
                        all_links_by_session.update(loaded_links) # Merge existing links
                    print(f"Loaded existing presentation links from {links_file} for partial resume.")
                except json.JSONDecodeError:
                     print(f"Error decoding {links_file}. Starting Stage 2 from scratch.")
                     all_links_by_session = {}


            processed_sessions = set(all_links_by_session.keys())
            sessions_to_process = [url for url in session_urls if url not in processed_sessions]

            if not sessions_to_process and os.path.exists(links_file): # Check if all sessions processed AND file exists
                 print("✅ All session links already processed for this program based on existing file.")
            else:
                print(f"ℹ️ Processing {len(sessions_to_process)} / {len(session_urls)} sessions.")
                # Re-initialize driver for Stage 2
                options = Options()
                options.add_argument("--headless")
                options.add_argument("--no-sandbox")
                options.add_argument("--remote-debugging-pipe")
                # Added this option to help with WebDriverException in Colab
                options.add_argument("--disable-dev-shm-usage")
                driver = webdriver.Chrome(options=options)
                try:
                    for sidx, session_url in enumerate(sessions_to_process, start=1):
                        print(f"   - Processing Session {sidx}/{len(sessions_to_process)}: {session_url}")
                        links = extract_presentation_links_from_live_page(driver, session_url) # Pass driver
                        all_links_by_session[session_url] = links
                        print(f"     -> {len(links)} links")
                        # Removed periodic save here


                finally:
                     driver.quit()

            # Final save at the end of Stage 2 if it wasn't skipped
            if not links_file_exists:
                 with open(links_file, "w", encoding="utf-8") as f:
                     json.dump(all_links_by_session, f, indent=2)
                 print(f"✅ Stage 2 saved -> {links_file}")


        # Stage 3: Scraping paper details (skip if papers_file exists and is complete)
        print("[Stage 3] Scraping paper details...")
        papers_file_exists = os.path.exists(papers_file)

        if papers_file_exists:
            try:
                # Attempt to load the file to confirm it's valid JSON and truly completed
                with open(papers_file, "r", encoding="utf-8") as f:
                     existing_papers = json.load(f)
                # Check if the number of papers in the file matches the number of links from Stage 2
                # This is a more robust check for completion than just file existence
                links_from_stage2_count = 0
                if os.path.exists(links_file):
                     try:
                         with open(links_file, "r", encoding="utf-8") as f:
                             stage2_links_data = json.load(f)
                             for session_url, link_list in stage2_links_data.items():
                                 links_from_stage2_count += len(link_list)
                     except json.JSONDecodeError:
                         print(f"❌ Error decoding {links_file}. Cannot verify Stage 3 completion count.")
                         links_from_stage2_count = -1 # Indicate an issue


                if links_from_stage2_count != -1 and len(existing_papers) >= links_from_stage2_count:
                    print(f"✅ Stage 3 file found and appears complete: {papers_file} ({len(existing_papers)} papers). Skipping Stage 3 for this program.")
                    # If the file exists, is valid, and the number of papers matches or exceeds Stage 2 links, skip
                    continue # Move to the next program_url in the main loop
                else:
                     print(f"ℹ️ Stage 3 file found ({len(existing_papers)} papers) but does not appear complete (expected at least {links_from_stage2_count} papers). Proceeding to resume Stage 3.")
                     # If the file exists but is incomplete, proceed with resume logic below

            except json.JSONDecodeError:
                 print(f"❌ Error decoding {papers_file}. File might be incomplete. Proceeding to re-scrape/resume Stage 3.")
                 # If decoding fails, we assume the file is incomplete and proceed with scraping/resuming
                 papers_file_exists = False # Ensure we don't skip based on the failed load


        # If Stage 3 wasn't skipped (either file didn't exist, decode failed, or file was incomplete)
        # Load links from Stage 2 output (either newly scraped or loaded from file)
        # Ensure all_links_by_session is available, load from file if Stage 2 was skipped but Stage 3 wasn't
        if 'all_links_by_session' not in locals() and os.path.exists(links_file):
             try:
                 with open(links_file, "r", encoding="utf-8") as f:
                     all_links_by_session = json.load(f)
                 print(f"Loaded Stage 2 links from {links_file} for Stage 3 processing.")
             except json.JSONDecodeError:
                  print(f"❌ Error decoding {links_file}. Cannot load Stage 2 links for Stage 3.")
                  all_links_by_session = {} # Initialize empty to avoid error

        links = load_all_links(links_file) # This function needs the path to the links file

        # Load existing paper data if the file exists (for resuming if not skipping the whole stage)
        existing_papers = [] # Initialize empty for scraping
        if os.path.exists(papers_file): # This check is now primarily for loading data to resume within the stage if needed
             try:
                 with open(papers_file, "r", encoding="utf-8") as f:
                     existing_papers = json.load(f)
                 print(f"Loaded existing paper data from {papers_file} for partial resume.")
             except json.JSONDecodeError:
                 print(f"Error decoding {papers_file}. Starting Stage 3 from scratch.")
                 existing_papers = []


        processed_paper_urls = {paper["url"] for paper in existing_papers}
        paper_links_to_process = [link for link in links if link not in processed_paper_urls]

        if not paper_links_to_process and os.path.exists(papers_file): # Check if all papers processed AND file exists
             print("✅ All paper links already processed for this program based on existing file.")
             # Since all papers are processed and the file exists, we can explicitly move to the next program URL
             continue # Move to the next program_url in the main loop

        else: # If there are papers to process or the file didn't exist/decode failed
             print(f"ℹ️ Processing {len(paper_links_to_process)} / {len(links)} papers.")
             # Need to load existing data first, then append new data - Handled by 'all_data = existing_papers' below
             all_data = existing_papers # Start with already processed data
             options = Options() # Re-initialize driver options
             options.add_argument("--headless")
             options.add_argument("--no-sandbox")
             options.add_argument("--remote-debugging-pipe")
             # Added this option to help with WebDriverException in Colab
             options.add_argument("--disable-dev-shm-usage")
             driver = webdriver.Chrome(options=options)

             try:
                 for idx, url in enumerate(paper_links_to_process, start=1):
                     print(f"\n🔍 Processing ({idx}/{len(paper_links_to_process)}): {url}")
                     try:
                         driver.get(url)
                         WebDriverWait(driver, 20).until(
                             EC.any_of(
                                 EC.presence_of_element_located((By.CSS_SELECTOR, "section.titleContent")),
                                 EC.presence_of_element_located((By.CSS_SELECTOR, "div.field_Abstract"))
                             )
                         )
                         time.sleep(2)
                         soup = BeautifulSoup(driver.page_source, 'html.parser')

                         topic = get_text(soup, "p.favoriteItem")
                         date_time = get_text(soup, 'span.defaultTZ')
                         abstract = get_text(soup, 'section.field_Abstract')

                         presenting_author = ""
                         authors = []
                         authors_structured = []

                         person_list = soup.select_one(".PersonList")
                         if person_list:
                             for sec in person_list.find_all("section", recursive=False):
                                 h = sec.find("h5")
                                 if not h:
                                     continue
                                 head = h.get_text(strip=True)
                                 if head.lower().startswith("presenting"):
                                     designation = "Presenting Author"
                                 else:
                                     designation = "Author"

                                 for li in sec.select("li.RoleListItem"):
                                     name_tag = li.select_one("a")
                                     affil_tag = li.select_one('span.roleAffiliation li') or li.select_one("span.roleAffiliation")
                                     if not name_tag:
                                         continue
                                     name = name_tag.get_text(strip=True)
                                     affil = affil_tag.get_text(strip=True) if affil_tag else ""
                                     authors_structured.append({
                                         "name": name,
                                         "affiliation": affil,
                                         "designation": designation
                                     })

                             pa = next((a for a in authors_structured if a["designation"] == "Presenting Author"), None)
                             if pa:
                                 presenting_author = f'{pa["name"]} | {pa["affiliation"]}'.strip()

                             authors = [
                                 f'{a["name"]} | {a["affiliation"]}'.strip()
                                 for a in authors_structured
                                 if a["designation"] == "Author"
                                 and f'{a["name"]} | {a["affiliation"]}'.strip() != presenting_author
                             ]

                         if not authors_structured:
                             all_authors_elements = soup.select('li.RoleListItem')
                             temp_list = []
                             for author_el in all_authors_elements:
                                 name_tag = author_el.select_one('a')
                                 affil_tag = author_el.select_one('span.roleAffiliation')
                                 if name_tag:
                                     name = name_tag.get_text(strip=True)
                                     affil = affil_tag.get_text(strip=True) if affil_tag else ""
                                     temp_list.append({"name": name, "affiliation": affil, "designation": "Author"})
                             authors_structured = temp_list

                         if not presenting_author:
                             presenting_author_name = get_text(soup, 'a.presenter')
                             presenter_a = soup.select_one('a.presenter')
                             presenter_affil = ""
                             if presenter_a:
                                 li_parent = presenter_a.find_parent('li', class_='RoleListItem')
                                 if li_parent:
                                     affil_li = li_parent.select_one('span.roleAffiliation li') or li_parent.select_one('span.roleAffiliation')
                                     presenter_affil = affil_li.get_text(strip=True) if affil_li else ""
                             if not presenter_affil:
                                 presenter_affil = get_text(soup, 'span.roleAffiliation')
                             presenting_author = f"{presenting_author_name} | {presenter_affil}".strip()

                         if not authors:
                             authors = [
                                 f'{a["name"]} | {a["affiliation"]}'.strip()
                                 for a in authors_structured
                                 if f'{a["name"]} | {a["affiliation"]}'.strip() != presenting_author
                             ]

                         data = {
                             "url": url,
                             "topic": topic,
                             "date_time": date_time,
                             "abstract": abstract,
                             "presenting_author": presenting_author,
                             "authors": authors,
                             "authors_structured": authors_structured
                         }

                         all_data.append(data)
                         print("✅ Extracted successfully.")
                         print("📝 Abstract Preview:")
                         print("\n".join((data["abstract"] or "").splitlines()[:2]))


                         # Removed periodic save here


                     except Exception as e:
                         print(f"❌ Failed to extract from {url}\n   Error: {e}")

                 finally:
                     driver.quit()

                 # Final save at the end of processing remaining links for this program ID
                 output_dir = os.path.dirname(papers_file)
                 if output_dir:
                     os.makedirs(output_dir, exist_ok=True)

                 with open(papers_file, 'w', encoding='utf-8') as f:
                     json.dump(all_data, f, ensure_ascii=False, indent=2)

                 print(f"\n✅ Final Stage 3 save -> {papers_file}")


    print("\n🎉 Done for all Program URLs.")

if __name__ == "__main__":
    main()